**AutoTrust Motors - Used Cars Dataset**


# 1. Import Libraries


In [1]:
import pandas as pd
import numpy as np

# 2. Download Dataset

In [2]:
df = pd.read_csv("/content/ds_salaries.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (607, 12)


,Unnamed: 0,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


Remove unwanted Unnamed columns

In [3]:
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

print("Dataset loaded successfully!")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Shape: (607, 11)


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


# 3. Missing Values Per Column


In [4]:
print("Missing Values Per Column:")
print(df.isnull().sum())

Missing Values Per Column:
work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64


# 4. Data Types

In [5]:
print("\nData Types:")
print(df.dtypes)


Data Types:
work_year              int64
experience_level      object
employment_type       object
job_title             object
salary                 int64
salary_currency       object
salary_in_usd          int64
employee_residence    object
remote_ratio           int64
company_location      object
company_size          object
dtype: object


# 5. Identify Numerical and Categorical Columns

In [6]:
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

categorical_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("\nNumerical Columns:")
print(numerical_columns)

print("\nCategorical Columns:")
print(categorical_columns)


Numerical Columns:
['work_year', 'salary', 'salary_in_usd', 'remote_ratio']

Categorical Columns:
['experience_level', 'employment_type', 'job_title', 'salary_currency', 'employee_residence', 'company_location', 'company_size']


# 6. Create Data Quality Summary


In [7]:
quality_summary = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Missing Values": df.isnull().sum().values
})

# Initial classification based on stored data type
quality_summary["Initial Type"] = quality_summary["Column"].apply(
    lambda x: "Numerical"
    if x in numerical_columns
    else "Categorical"
)

print("\nData Quality Summary:")
display(quality_summary)


Data Quality Summary:


,Column,Data Type,Missing Values,Initial Type
0,work_year,int64,0,Numerical
1,experience_level,object,0,Categorical
2,employment_type,object,0,Categorical
3,job_title,object,0,Categorical
4,salary,int64,0,Numerical
5,salary_currency,object,0,Categorical
6,salary_in_usd,int64,0,Numerical
7,employee_residence,object,0,Categorical
8,remote_ratio,int64,0,Numerical
9,company_location,object,0,Categorical


In [8]:
# Make a copy of the original dataset
cleaned_df = df.copy()

#7. Check Duplicates

In [9]:
df.duplicated().sum()

np.int64(42)

# 8. Standardize Values

In [10]:
categorical_columns = cleaned_df.select_dtypes(
    include=["object", "string"]
).columns

for column in categorical_columns:

    # Remove extra spaces
    cleaned_df[column] = cleaned_df[column].str.strip()

print("Categorical values standardized successfully.")

Categorical values standardized successfully.


#9. Unique Values

In [11]:
for column in categorical_columns:
    print("\n", column)
    print(cleaned_df[column].unique())


 experience_level
['MI' 'SE' 'EN' 'EX']

 employment_type
['FT' 'CT' 'PT' 'FL']

 job_title
['Data Scientist' 'Machine Learning Scientist' 'Big Data Engineer'
 'Product Data Analyst' 'Machine Learning Engineer' 'Data Analyst'
 'Lead Data Scientist' 'Business Data Analyst' 'Lead Data Engineer'
 'Lead Data Analyst' 'Data Engineer' 'Data Science Consultant'
 'BI Data Analyst' 'Director of Data Science' 'Research Scientist'
 'Machine Learning Manager' 'Data Engineering Manager'
 'Machine Learning Infrastructure Engineer' 'ML Engineer' 'AI Scientist'
 'Computer Vision Engineer' 'Principal Data Scientist'
 'Data Science Manager' 'Head of Data' '3D Computer Vision Researcher'
 'Data Analytics Engineer' 'Applied Data Scientist'
 'Marketing Data Analyst' 'Cloud Data Engineer' 'Financial Data Analyst'
 'Computer Vision Software Engineer' 'Director of Data Engineering'
 'Data Science Engineer' 'Principal Data Engineer'
 'Machine Learning Developer' 'Applied Machine Learning Scientist'
 'Data Ana

# 10. Detect and Handle Outliers

In [12]:

numerical_columns = cleaned_df.select_dtypes(
    include=["int64", "float64"]
).columns

outlier_summary = []

for column in numerical_columns:

    Q1 = cleaned_df[column].quantile(0.25)
    Q3 = cleaned_df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = cleaned_df[
        (cleaned_df[column] < lower_bound) |
        (cleaned_df[column] > upper_bound)
    ]

    outlier_summary.append({
        "Column": column,
        "Number of Outliers": len(outliers),
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound
    })

outlier_summary = pd.DataFrame(outlier_summary)

print("Outlier Summary:")
display(outlier_summary)

Outlier Summary:


,Column,Number of Outliers,Lower Bound,Upper Bound
0,work_year,0,2019.5,2023.5
1,salary,44,-72500.0,307500.0
2,salary_in_usd,10,-68185.0,280911.0
3,remote_ratio,0,-25.0,175.0


# handle outliers

In [13]:

print("Outlier treatment:")

for column in numerical_columns:

    Q1 = cleaned_df[column].quantile(0.25)
    Q3 = cleaned_df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (cleaned_df[column] < lower_bound) |
        (cleaned_df[column] > upper_bound)
    ).sum()

    print(f"{column}: {outlier_count} outliers detected.")

print("\nOutliers were inspected and no values were removed unless they were confirmed as data-entry errors.")

Outlier treatment:
work_year: 0 outliers detected.
salary: 44 outliers detected.
salary_in_usd: 10 outliers detected.
remote_ratio: 0 outliers detected.

Outliers were inspected and no values were removed unless they were confirmed as data-entry errors.


# 11. Prepare Clean Dataset

In [14]:
# Reset index after cleaning
cleaned_df = cleaned_df.reset_index(drop=True)

print("Clean Dataset Shape:", cleaned_df.shape)

print("\nMissing Values:")
print(cleaned_df.isnull().sum())

print("\nDuplicate Rows:")
print(cleaned_df.duplicated().sum())

print("\nClean Dataset:")
display(cleaned_df.head())

Clean Dataset Shape: (607, 11)

Missing Values:
work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64

Duplicate Rows:
42

Clean Dataset:


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2020,MI,FT,Data Scientist,70000,EUR,79833,DE,0,DE,L
1,2020,SE,FT,Machine Learning Scientist,260000,USD,260000,JP,0,JP,S
2,2020,SE,FT,Big Data Engineer,85000,GBP,109024,GB,50,GB,M
3,2020,MI,FT,Product Data Analyst,20000,USD,20000,HN,0,HN,S
4,2020,SE,FT,Machine Learning Engineer,150000,USD,150000,US,50,US,L


# Save Clean Dataset

In [15]:

cleaned_df.to_csv(
    "/content/cleaned_dataset.csv",
    index=False
)

print("Cleaned Dataset saved successfully!")

Cleaned Dataset saved successfully!


In [16]:
# ============================================================
# DATA DICTIONARY
# ============================================================

data_dictionary = pd.DataFrame({
    "Column": cleaned_df.columns,

    "Data Type": [
        str(cleaned_df[column].dtype)
        for column in cleaned_df.columns
    ],

    "Number of Rows": [
        cleaned_df[column].shape[0]
        for column in cleaned_df.columns
    ],

    "Missing Values": [
        cleaned_df[column].isnull().sum()
        for column in cleaned_df.columns
    ],

    "Unique Values": [
        cleaned_df[column].nunique()
        for column in cleaned_df.columns
    ],

    "Description": [
        "Year the salary was reported"
        if column == "work_year"

        else "Employee experience level"
        if column == "experience_level"

        else "Type of employment"
        if column == "employment_type"

        else "Employee job title"
        if column == "job_title"

        else "Reported salary in the original currency"
        if column == "salary"

        else "Currency used for the reported salary"
        if column == "salary_currency"

        else "Salary converted to US dollars"
        if column == "salary_in_usd"

        else "Country where the employee resides"
        if column == "employee_residence"

        else "Percentage of work performed remotely"
        if column == "remote_ratio"

        else "Country where the company is located"
        if column == "company_location"

        else "Company size"
        if column == "company_size"

        else "Description not available"
        for column in cleaned_df.columns
    ]
})

# Display Data Dictionary

print("Data Dictionary:")
display(data_dictionary)


# SAVE DATA DICTIONARY

data_dictionary.to_csv(
    "/content/data_dictionary.csv",
    index=False
)

print("Data Dictionary saved successfully!")

Data Dictionary:


,Column,Data Type,Number of Rows,Missing Values,Unique Values,Description
0,work_year,int64,607,0,3,Year the salary was reported
1,experience_level,object,607,0,4,Employee experience level
2,employment_type,object,607,0,4,Type of employment
3,job_title,object,607,0,50,Employee job title
4,salary,int64,607,0,272,Reported salary in the original currency
5,salary_currency,object,607,0,17,Currency used for the reported salary
6,salary_in_usd,int64,607,0,369,Salary converted to US dollars
7,employee_residence,object,607,0,57,Country where the employee resides
8,remote_ratio,int64,607,0,3,Percentage of work performed remotely
9,company_location,object,607,0,50,Country where the company is located


Data Dictionary saved successfully!
